In [1]:
import refinitiv.data as rd
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

rd.open_session()

<refinitiv.data.session.Definition object at 0x11e2ee9c0 {name='workspace'}>

In [11]:
ALKU = "2010-01-01"
LOPPU = "2026-04-01"

In [ ]:
# --- Indeksin jäsenlista (.STOXX) ---
df = rd.get_history(
    universe=[".STOXX"],
    fields=[

"TR.IndexJLConstituentComName",
    ],
    start=ALKU,
    end=LOPPU,
    interval="1D",
)

In [13]:
df.head(100000)

.STOXX,Constituent Common Name
Date,
2010-02-19,Cadbury Ltd
2010-03-23,Corporacion Financiera Alba SA
2010-03-23,Buzzi SpA
2010-03-23,BKW Energie AG
2010-03-23,Volkswagen AG
...,...
2026-03-23,Greggs PLC
2026-03-23,Grafton Group PLC
2026-03-23,Alten SA


In [19]:
import refinitiv.data as rd
import pandas as pd

df = rd.get_history(
    universe=[".STOXX"],
    fields=["TR.IndexJLConstituentRIC"],
    start=ALKU,
    end=LOPPU,
    interval="1D",
)

# reset index → päivämäärä sarakkeeksi
df = df.reset_index()

df.head()

.STOXX,Date,Constituent RIC
0,2010-02-19,CBRY.L^C10
1,2010-03-23,ALB.MC^E25
2,2010-03-23,BZU.MI
3,2010-03-23,BKWN.S^D12
4,2010-03-23,VOWG.DE


In [20]:
# ryhmittele per päivä → setti yhtiöitä
daily = (
    df.groupby("Date")["Constituent RIC"]
    .apply(lambda x: set(x.dropna()))
    .sort_index()
)

changes = []

dates = daily.index

for i in range(1, len(dates)):
    prev = daily.iloc[i - 1]
    curr = daily.iloc[i]

    added = curr - prev
    removed = prev - curr

    if added or removed:
        changes.append({
            "date": dates[i],
            "added": list(added),
            "removed": list(removed)
        })

changes_df = pd.DataFrame(changes)
changes_df

,date,added,removed
0,2010-03-23,"[BZU.MI, GB3304011.L^H16, NOS.LS, BKWN.S^D12, ...",[CBRY.L^C10]
1,2010-03-29,[LUN.CO^F22],"[BZU.MI, GB3304011.L^H16, NOS.LS, BKWN.S^D12, ..."
2,2010-03-30,[CPWN.L^C10],[LUN.CO^F22]
3,2010-03-31,[TALK.L^C21],[CPWN.L^C10]
4,2010-04-28,[ENQ.L],[TALK.L^C21]
...,...,...,...
286,2025-10-01,[MDBI.MI],"[EMBRACb.ST, CCC.L, VRLA.PA, RDC.DE, AFXG.DE, ..."
287,2025-10-02,[ROO.L^J25],[MDBI.MI]
288,2025-10-08,[TKWY.AS^K25],[ROO.L^J25]
289,2025-12-22,"[CLN.S, OCDO.L, IFCN.S, LXSG.DE, DIAS.MI, TKMS...",[TKWY.AS^K25]


In [23]:
import refinitiv.data as rd
import pandas as pd

ALKU = "2010-01-01"
LOPPU = "2026-03-31"

rd.open_session()

df = rd.get_history(
    universe=[".STOXX"],
    fields=[
        "TR.IndexJLConstituentRIC",
        "TR.IndexJLConstituentComName",
    ],
    start=ALKU,
    end=LOPPU,
    interval="1D",
)

df = df.reset_index()

ric_col = "Constituent RIC"
name_col = "Constituent Common Name"

df = df.dropna(subset=[ric_col]).copy()
df["Date"] = pd.to_datetime(df["Date"]).dt.normalize()

# Poista duplikaatit päivän sisällä
df = df.drop_duplicates(subset=["Date", ric_col])

# Päivittäiset RIC-setit
daily_rics = (
    df.groupby("Date")[ric_col]
    .apply(lambda x: set(x.dropna().unique()))
    .sort_index()
)

# Debug (voit halutessa jättää pois)
counts = daily_rics.apply(len)
print("Constituents per day:")
print(counts.describe())

# Nimi mapping (viimeisin nimi per RIC)
name_map = (
    df.sort_values("Date")
      .dropna(subset=[ric_col, name_col])
      .drop_duplicates(subset=[ric_col], keep="last")
      .set_index(ric_col)[name_col]
      .to_dict()
)

# Päiväkohtainen nimi (tarkempi)
daily_name_map = (
    df.dropna(subset=[ric_col, name_col])
      .drop_duplicates(subset=["Date", ric_col], keep="last")
      .set_index(["Date", ric_col])[name_col]
      .to_dict()
)

dates = list(daily_rics.index)
change_rows = []

for i in range(1, len(dates)):
    prev_date = dates[i - 1]
    curr_date = dates[i]

    prev_set = daily_rics.loc[prev_date]
    curr_set = daily_rics.loc[curr_date]

    added = sorted(curr_set - prev_set)
    removed = sorted(prev_set - curr_set)

    for ric in added:
        change_rows.append({
            "date": curr_date,
            "ric": ric,
            "company": daily_name_map.get((curr_date, ric), name_map.get(ric, ric)),
            "action": "added",
        })

    for ric in removed:
        change_rows.append({
            "date": curr_date,
            "ric": ric,
            "company": daily_name_map.get((prev_date, ric), name_map.get(ric, ric)),
            "action": "removed",
        })

changes_df = pd.DataFrame(change_rows).sort_values(["date", "action", "ric"])

print(changes_df.head(50))

changes_df.to_csv("stoxx_membership_changes.csv", index=False)



Constituents per day:
count    292.000000
mean       2.784247
std        3.669980
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       22.000000
Name: Constituent RIC, dtype: float64
         date              ric                           company   action
0  2010-03-23       ALB.MC^E25    Corporacion Financiera Alba SA    added
1  2010-03-23       BKWN.S^D12                    BKW Energie AG    added
2  2010-03-23        BRE.L^D11        Brit Insurance Holdings BV    added
3  2010-03-23           BZU.MI                         Buzzi SpA    added
4  2010-03-23       EIGE.L^C20                      Ei Group Ltd    added
5  2010-03-23  GB3304011.L^H16                         Darty Ltd    added
6  2010-03-23          GMAB.CO                        Genmab A/S    added
7  2010-03-23      GTCH.MI^D15            Brightstar Lottery PLC    added
8  2010-03-23           NOS.LS                       NOS SGPS SA    added
9  2010-03-23      QCEG.DE^J12          